In [ ]:
# ==========================================================
# CELL 1 — SETTINGS (change numbers here, nothing else)
# ==========================================================

HOW_MANY_DIALOGUES = 1       # how many conversations to test (max 20)
HOW_MANY_QUESTIONS_EACH = 5   # how many questions per conversation (max 20)

In [ ]:
# ==========================================================
# CELL 2 — Setup (load model once)
# ==========================================================
from brain import Brain
from agent import Agent
from tracer import Tracer
from tools import ALL_TOOLS
from memory.memory_manager import MemoryManager
import config

brain_gpu0 = Brain(TOKEN, device="cuda:0")
brain_gpu1 = Brain(TOKEN, device="cuda:1")

def make_generate_fn(brain_instance):
    def generate(prompt):
        return brain_instance.think([{"role": "user", "content": prompt}])["text"]
    return generate

local_generate_0 = make_generate_fn(brain_gpu0)
local_generate_1 = make_generate_fn(brain_gpu1)

In [ ]:
# ==========================================================
# CELL 3 — Load the dataset (plain, no groupby)
# ==========================================================
import pandas as pd

all_questions = pd.read_csv("beam_100k_questions.csv")
all_histories = pd.read_csv("beam_100k_histories.csv")

# get the list of conversation IDs, and only keep as many as we want to test
conversation_id_list = all_histories["conversation_id"].unique()
conversation_id_list = conversation_id_list[:HOW_MANY_DIALOGUES]

print("Testing", len(conversation_id_list), "conversations")

In [ ]:
# ==========================================================
# CELL 4 — Judge function (checks if the answer is correct)
# ==========================================================
import ast

def judge_abstention(agent_answer):
    answer_lower = agent_answer.lower()
    for marker in config.ABSTENTION_MARKERS:
        if marker in answer_lower:
            return True
    return False

def judge_with_rubric(agent_answer, rubric_text):
    try:
        rubric_list = ast.literal_eval(rubric_text)
    except Exception:
        rubric_list = [rubric_text]

    answer_lower = agent_answer.lower()
    matches = 0
    for point in rubric_list:
        if str(point).lower()[:20] in answer_lower:
            matches = matches + 1

    needed = max(1, len(rubric_list) // 2)
    return matches >= needed

def judge(question_type, agent_answer, rubric_text):
    if question_type == "abstention":
        return judge_abstention(agent_answer)
    else:
        return judge_with_rubric(agent_answer, rubric_text)

In [ ]:
# ==========================================================
# CELL 5 (FIXED) — Main loop, TRUE parallel across 2 GPUs using PROCESSES
# ==========================================================
# WHY THIS CHANGED FROM threading TO multiprocessing:
# Python threads share one GIL, so your two "GPU threads" were taking turns,
# not running at the same time -> that's why you saw 1, 11, 2, 12, ... in
# lockstep instead of both GPUs finishing independently.
# multiprocessing.Process gives each GPU its own real interpreter -> true parallel work.

import os
import time
import signal
import multiprocessing as mp
import pandas as pd

QUESTION_TIMEOUT_SECONDS = 90          # kill a single question if it hangs this long
SETUP_TIMEOUT_SECONDS = 300            # kill memory setup (extraction) if it hangs this long


class TimeoutError_(Exception):
    pass


def _raise_timeout(signum, frame):
    raise TimeoutError_("timed out")


def run_with_timeout(fn, seconds, *args, **kwargs):
    """Runs fn(*args, **kwargs) but gives up after `seconds`.
    Only works in a process's MAIN thread (true here, since each GPU worker
    is its own process)."""
    old_handler = signal.signal(signal.SIGALRM, _raise_timeout)
    signal.alarm(seconds)
    try:
        return fn(*args, **kwargs)
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)


def run_some_dialogues(dialogue_ids, device, gpu_label, project_dir,
                        all_histories_path, all_questions_path,
                        how_many_questions_each, hf_token, results_file):

    import sys
    sys.path.insert(0, project_dir)

    # FIX: everything CUDA-related is created INSIDE the process, not passed in
    # from the parent. CUDA contexts don't survive being handed to a child process.
    from brain import Brain
    from agent import Agent
    from tracer import Tracer
    from tools import ALL_TOOLS
    from memory.memory_manager import MemoryManager
    from sentence_transformers import SentenceTransformer
    import config
    import shutil
    import ast

    all_histories = pd.read_csv(all_histories_path)
    all_questions = pd.read_csv(all_questions_path)

    brain_instance = Brain(hf_token, device=device)

    def generate_fn(prompt):
        return brain_instance.think([{"role": "user", "content": prompt}])["text"]

    # FIX: embedder loaded ONCE per process (per GPU), reused for every
    # conversation this process handles, instead of reloaded from scratch
    # each time a new MemoryManager() is made.
    shared_embedder = SentenceTransformer(config.EMBEDDING_MODEL)

    def judge_abstention(agent_answer):
        answer_lower = agent_answer.lower()
        for marker in config.ABSTENTION_MARKERS:
            if marker in answer_lower:
                return True
        return False

    def judge_with_rubric(agent_answer, rubric_text):
        try:
            rubric_list = ast.literal_eval(rubric_text)
        except Exception:
            rubric_list = [rubric_text]
        answer_lower = agent_answer.lower()
        matches = 0
        for point in rubric_list:
            if str(point).lower()[:20] in answer_lower:
                matches = matches + 1
        needed = max(1, len(rubric_list) // 2)
        return matches >= needed

    def judge(question_type, agent_answer, rubric_text):
        if question_type == "abstention":
            return judge_abstention(agent_answer)
        return judge_with_rubric(agent_answer, rubric_text)

    base_folder = config.DATA_FOLDER

    for conversation_id in dialogue_ids:
        print("\n=== [" + gpu_label + "] Conversation:", conversation_id, "===", flush=True)

        conv_folder = base_folder + "/conv_" + str(conversation_id)
        if os.path.exists(conv_folder):
            shutil.rmtree(conv_folder)
        os.makedirs(conv_folder, exist_ok=True)

        memory = MemoryManager(embedder=shared_embedder)
        memory.historical.file_path = conv_folder + "/history.json"
        memory.semantic.file_path = conv_folder + "/facts.json"
        memory.episodic.file_path = conv_folder + "/episodes.json"

        tracer = Tracer()
        agent = Agent(brain_instance, memory, ALL_TOOLS, tracer)

        history_row = all_histories[all_histories["conversation_id"] == conversation_id].iloc[0]
        full_text = history_row["conversation"]

        # FIX: setup (fact extraction) is now timed AND has its own timeout,
        # instead of running unmeasured and unbounded.
        setup_start = time.time()
        try:
            run_with_timeout(
                memory.add_to_historical, SETUP_TIMEOUT_SECONDS,
                full_text, generate_fn=generate_fn, auto_extract=True,
            )
        except TimeoutError_:
            print(f"  [{gpu_label}] SETUP TIMED OUT on conversation {conversation_id}, skipping it.", flush=True)
            continue
        except Exception as e:
            print(f"  [{gpu_label}] SETUP FAILED on conversation {conversation_id}: {e}, skipping it.", flush=True)
            continue
        setup_seconds = round(time.time() - setup_start, 2)
        print(f"  [{gpu_label}] setup took {setup_seconds}s", flush=True)

        this_conversation_questions = all_questions[all_questions["conversation_id"] == conversation_id]
        this_conversation_questions = this_conversation_questions.head(how_many_questions_each)

        for row_number in range(len(this_conversation_questions)):
            row = this_conversation_questions.iloc[row_number]
            question_text = row["question"]
            question_type = row["question_type"]
            rubric_text = row["rubric"]

            start_time = time.time()
            try:
                # FIX: real per-question timeout. A single hung question can
                # no longer freeze the whole run.
                agent_answer = run_with_timeout(agent.ask, QUESTION_TIMEOUT_SECONDS, question_text)
            except TimeoutError_:
                agent_answer = "[TIMED OUT]"
                print(f"  [{gpu_label}][{question_type}] TIMED OUT after {QUESTION_TIMEOUT_SECONDS}s", flush=True)
            except Exception as e:
                agent_answer = f"[ERROR: {e}]"
                print(f"  [{gpu_label}][{question_type}] ERROR: {e}", flush=True)
            time_taken = time.time() - start_time

            is_correct = judge(question_type, agent_answer, rubric_text) if not agent_answer.startswith("[") else False
            print("  [" + gpu_label + "][" + question_type + "] correct =", is_correct, flush=True)

            one_result = pd.DataFrame([{
                "question_id": row["question_id"],
                "conversation_id": conversation_id,
                "q_type": question_type,
                "question": question_text,
                "gold_answer": row["gold_answer"],
                "agent_answer": agent_answer,
                "correct": is_correct,
                "seconds": round(time_taken, 2),
            }])

            # FIX: each process writes to ITS OWN file (results_file is unique
            # per GPU) -> no more two processes racing to append to one CSV.
            file_already_exists = os.path.exists(results_file)
            one_result.to_csv(results_file, index=False, mode="a", header=not file_already_exists)

    print(f"[{gpu_label}] finished all assigned conversations.", flush=True)


if __name__ == "__main__":
    # ── EDIT THESE ─────────────────────────────────────────────
    PROJECT_DIR = "/kaggle/working/Harness-Memory-"
    HISTORIES_CSV = "/kaggle/input/datasets/sadi2oo3q/beam-dataset-100k/beam_100k_histories.csv"
    QUESTIONS_CSV = "/kaggle/input/datasets/sadi2oo3q/beam-dataset-100k/beam_100k_questions.csv"
    HOW_MANY_DIALOGUES = 20
    HOW_MANY_QUESTIONS_EACH = 10
    # ────────────────────────────────────────────────────────────

    # FIX: required for CUDA + multiprocessing to work together safely.
    mp.set_start_method("spawn", force=True)

    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")

    all_histories = pd.read_csv(HISTORIES_CSV)
    conversation_id_list = list(all_histories["conversation_id"].unique())[:HOW_MANY_DIALOGUES]

    half_point = len(conversation_id_list) // 2
    first_half = conversation_id_list[:half_point]
    second_half = conversation_id_list[half_point:]

    print("GPU 0 will do", len(first_half), "conversations")
    print("GPU 1 will do", len(second_half), "conversations")

    p0 = mp.Process(target=run_some_dialogues, args=(
        first_half, "cuda:0", "GPU0", PROJECT_DIR, HISTORIES_CSV, QUESTIONS_CSV,
        HOW_MANY_QUESTIONS_EACH, hf_token, "beam_eval_results_gpu0.csv",
    ))
    p1 = mp.Process(target=run_some_dialogues, args=(
        second_half, "cuda:1", "GPU1", PROJECT_DIR, HISTORIES_CSV, QUESTIONS_CSV,
        HOW_MANY_QUESTIONS_EACH, hf_token, "beam_eval_results_gpu1.csv",
    ))

    p0.start()
    p1.start()
    p0.join()
    p1.join()

    print("\nDone. Merging results...")
    df0 = pd.read_csv("beam_eval_results_gpu0.csv") if os.path.exists("beam_eval_results_gpu0.csv") else pd.DataFrame()
    df1 = pd.read_csv("beam_eval_results_gpu1.csv") if os.path.exists("beam_eval_results_gpu1.csv") else pd.DataFrame()
    merged = pd.concat([df0, df1], ignore_index=True)
    merged.to_csv("beam_eval_results.csv", index=False)
    print("Overall accuracy:", round(merged["correct"].mean() * 100, 1), "%")

In [ ]:
# ==========================================================
# CELL 6 — See the score
# ==========================================================
results = pd.read_csv(results_file)

print("Overall accuracy:", round(results["correct"].mean() * 100, 1), "%")
print()
print(results.groupby("q_type")["correct"].mean())